# Adding Name Similarity for English

I'm exploring ways to look more closely at coverage issues. One idea: names tend to be relatively stable, so maybe we can measure the agreement between the glossed name for the lemma and the target text. Note this will only help for English (and perhaps Chinese): we don't have glosses for other data.

**BLOCKED**:
I didn't think this through well enough. 
* The WLCM source data indicates when a token has `pos='Name'`, but for SBLGNT it's a combination of `class='noun'` and `type='proper'`. I don't think this data is preserved in the `Source` dataclass.
    * FIXES:
        * design a coherent POS/morphology scheme across both testaments.
        * make sure `SourceReader` captures all the relevant information
* Even if this data were available, how to determine if the target token is a name? Capitalization isn't reliable because sentence-initial tokens are capitalized. We have a list of names from ACAI: we could read that in and check membership, but that introduces a questionable dependency on ACAI.

So this is left incomplete for now.

---

Exceptions: 
* pronoun/name alternation: we could be smart perhaps about allowing this?
* name variation (Israel vs Jacob): add exceptions list

In [1]:
from importlib import reload

from biblealignlib.burrito import Manager, AlignmentSet
from biblealignlib.coverage import CLEARROOT, CoverageAnalyzer, CoverageFilter
from biblealignlib.coverage import analyzer
targetlang = "eng"
alset = AlignmentSet(sourceid="SBLGNT", 
                     targetid="NIV11",
                     targetlanguage=targetlang,
                     langdatapath=(CLEARROOT / f"alignments-{targetlang}/data"))
mgr = Manager(alset)


        - sourcepath: /Users/sboisen/git/Clear-Bible/Alignments/data/sources/SBLGNT.tsv
        - targetpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/targets/NIV11/nt_NIV11.tsv
        - alignmentpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/alignments/NIV11/SBLGNT-NIV11-manual.json
        - tomlpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/alignments/NIV11/SBLGNT-NIV11-manual.toml
        
Dropping 2 bad alignment records. Instances in self.alignmentsreader.badrecords.
Target selector is included in multiple records	2


In [26]:
bcvid = "41004003"
mrk43 = mgr.bcv["versedata"][bcvid]
mrk43

<VerseData: 41004003>

In [29]:
reload(analyzer)

<module 'biblealignlib.coverage.analyzer' from '/Users/sboisen/git/Clear-Bible/biblealignlib/biblealignlib/coverage/analyzer.py'>

In [30]:
nsa = analyzer.NameSimilarityAnalyzer(mgr)

In [31]:
sc, tc = nsa._compute_token_coverage(mrk43)

In [35]:
for t in tc: print(t.should_count, t.token._display)

True 41004003002: Listen		 ('', False, False)
True 41004003006: went		 ('', False, False)
True 41004003007: out		 ('', False, False)
True 41004003009: sow		 ('', False, False)


In [34]:
nsa.verse_coverage(bcvid).asdict()

{'BCVID': '41004003',
 'Verse': '003',
 'Chapter': '41004',
 'Book': '41',
 'Reference': 'MRK 4:3',
 'Source_Total': 0,
 'Source_Aligned': 0,
 'Source_Unaligned': 0,
 'Source_Coverage_Pct': 0.0,
 'Target_Total': 4,
 'Target_Aligned': 0,
 'Target_Unaligned': 4,
 'Target_Coverage_Pct': 0.0}

In [6]:
STOP

NameError: name 'STOP' is not defined

In [42]:
for slist,tlist in mgr.bcv["versedata"]["41001001"].alignments:
    for s in slist:
        print(s.asdict())
    for t in tlist:
        print(t.asdict())
    print("-----")

{'id': 'n41001001001', 'altId': 'Ἀρχὴ-1', 'text': 'Ἀρχὴ', 'strongs': 'G0746', 'gloss': '[The] beginning', 'gloss2': 'beginning', 'lemma': 'ἀρχή', 'pos': 'noun', 'morph': 'N-NSF', 'required': True}
{'id': '41001001002', 'text': 'beginning', 'source_verse': '41001001', 'skip_space_after': '', 'exclude': ''}
-----
{'id': 'n41001001002', 'altId': 'τοῦ-1', 'text': 'τοῦ', 'strongs': 'G3588', 'gloss': 'of the', 'gloss2': 'the', 'lemma': 'ὁ', 'pos': 'det', 'morph': 'T-GSN', 'required': False}
{'id': '41001001004', 'text': 'the', 'source_verse': '41001001', 'skip_space_after': '', 'exclude': ''}
-----
{'id': 'n41001001003', 'altId': 'εὐαγγελίου-1', 'text': 'εὐαγγελίου', 'strongs': 'G2098', 'gloss': 'gospel', 'gloss2': 'gospel', 'lemma': 'εὐαγγέλιον', 'pos': 'noun', 'morph': 'N-GSN', 'required': True}
{'id': '41001001005', 'text': 'good', 'source_verse': '41001001', 'skip_space_after': '', 'exclude': ''}
{'id': '41001001006', 'text': 'news', 'source_verse': '41001001', 'skip_space_after': '', 'e

In [41]:
mgr.bcv["versedata"]["41001001"].alignments

[([<Source: 41001001001>], [<Target: 41001001002>]),
 ([<Source: 41001001002>], [<Target: 41001001004>]),
 ([<Source: 41001001003>], [<Target: 41001001005>, <Target: 41001001006>]),
 ([<Source: 41001001004>], [<Target: 41001001008>]),
 ([<Source: 41001001005>],
  [<Target: 41001001010>, <Target: 41001001013>, <Target: 41001001015>])]